In [ ]:
!pip install word2number

In [ ]:
!pip install pandas numpy scikit-learn word2number joblib

In [ ]:
import io
import pandas as pd
csv_data="""experience,test_score,interview_score,salary
,8,9,50000
,8,6,45000
five,6,7,60000
two,10,10,65000
seven,9,6,70000
ten,7,10,80000
eleven,7,7,86000"""
df = pd.read_csv(io.StringIO(csv_data))


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from word2number import w2n

# 1. Load Dataset
#df = pd.read_csv("hiring.csv")

# 2. Data Cleaning & Feature Handling
# Fill missing experience values with 'zero'
df["experience"] = df["experience"].fillna("zero")

# Convert word numbers to integer values (e.g., 'five' -> 5)
df["experience"] = df["experience"].apply(
    lambda x: w2n.word_to_num(str(x)) if isinstance(x, str) else x
)

# Impute missing test scores with the median value
median_test_score = df["test_score"].median()
df["test_score"] = df["test_score"].fillna(median_test_score)

# 3. Model 1: Single Feature Model (Experience only)
X_single = df[["experience"]]
y = df["salary"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_single, y, test_size=0.2, random_state=42
)

single_model = LinearRegression()
single_model.fit(X_train_s, y_train_s)
y_pred_s = single_model.predict(X_test_s)

rmse_single = np.sqrt(mean_squared_error(y_test_s, y_pred_s))
r2_single = r2_score(y_test_s, y_pred_s)

# 4. Model 2: Multiple Feature Model (Experience, Test Score, Interview Score)
X_multi = df[["experience", "test_score", "interview_score"]]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

multi_model = LinearRegression()
multi_model.fit(X_train_m, y_train_m)
y_pred_m = multi_model.predict(X_test_m)

rmse_multi = np.sqrt(mean_squared_error(y_test_m, y_pred_m))
r2_multi = r2_score(y_test_m, y_pred_m)

# 5. Compare Performance
print("=== Performance Comparison ===")
print(
    f"Single Feature Model  -> RMSE: {rmse_single:.2f} | R² Score: {r2_single:.4f}"
)
print(
    f"Multiple Feature Model -> RMSE: {rmse_multi:.2f} | R² Score: {r2_multi:.4f}"
)

# 6. Save the Best Model
best_model = multi_model if r2_multi >= r2_single else single_model
joblib.dump(best_model, "best_salary_model.pkl")
print("\nBest model saved successfully as 'best_salary_model.pkl'.")

# 7. Sample Predictions
# Predict salary for 2 years experience, 9 test score, 6 interview score
sample_candidate_1 = np.array([[2, 9, 6]])
pred_1 = multi_model.predict(sample_candidate_1)

# Predict salary for 12 years experience, 10 test score, 10 interview score
sample_candidate_2 = np.array([[12, 10, 10]])
pred_2 = multi_model.predict(sample_candidate_2)

print("\n=== Sample Predictions ===")
print(f"Candidate 1 (2 yrs exp, 9 test, 6 interview): ${pred_1[0]:,.2f}")
print(f"Candidate 2 (12 yrs exp, 10 test, 10 interview): ${pred_2[0]:,.2f}")

=== Performance Comparison ===
Single Feature Model  -> RMSE: 7133.88 | R² Score: -7.1428
Multiple Feature Model -> RMSE: 1859.30 | R² Score: 0.4469

Best model saved successfully as 'best_salary_model.pkl'.

=== Sample Predictions ===
Candidate 1 (2 yrs exp, 9 test, 6 interview): $56,286.68
Candidate 2 (12 yrs exp, 10 test, 10 interview): $96,010.50


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
